In [7]:
# nạp các hàm xử lý dữ liệu và đánh giá.

import os
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from IPython.display import display

def find_column(df, names):
    for n in df.columns:
        if n.lower() in [x.lower() for x in names]:
            return n
    return None

def generate_ratings_for_sheet(df, as_float=False, seed=42):
    np.random.seed(seed)
    working = df.copy()
    
    # Đảm bảo có các cột ID
    user_col = find_column(working, ['UserID', 'UserId', 'user_id', 'user'])
    course_col = find_column(working, ['CourseID', 'CourseId', 'course_id', 'course'])
    if user_col is None or course_col is None:
        raise ValueError('Không tìm thấy cột UserID hoặc CourseID trong dữ liệu')

    # Tính toán độ lệch (bias) dựa trên mức độ phổ biến của khóa học
    course_counts = working[course_col].value_counts().astype(float)
    cc_min, cc_max = course_counts.min(), course_counts.max()
    if cc_max - cc_min <= 0:
        norm_counts = course_counts * 0.0
    else:
        norm_counts = (course_counts - cc_min) / (cc_max - cc_min)

    working['_course_pop'] = working[course_col].map(norm_counts).fillna(0.0)

    # Tính toán độ lệch dựa trên mức độ hoạt động của user
    user_counts = working[user_col].value_counts().astype(float)
    uc_min, uc_max = user_counts.min(), user_counts.max()
    if uc_max - uc_min <= 0:
        norm_users = user_counts * 0.0
    else:
        norm_users = (user_counts - uc_min) / (uc_max - uc_min)
    working['_user_act'] = working[user_col].map(norm_users).fillna(0.0)

    # Base mean rating và effect sizes
    base = 3.2
    course_effect = working['_course_pop'] * 1.2  # Tăng tối đa +1.2
    user_effect = (working['_user_act'] - 0.5) * 0.6  # Nằm trong khoảng [-0.3, +0.3]

    raw_means = base + course_effect + user_effect

    # Thêm nhiễu ngẫu nhiên và giới hạn trong khoảng 1..5
    noise = np.random.normal(0, 0.6, size=len(working))
    raw = raw_means + noise
    
    ratings = np.clip(np.round(raw, 2) if as_float else np.rint(raw), 1, 5)

    if as_float:
        working['Rating'] = ratings.astype(float)
    else:
        working['Rating'] = ratings.astype(int)

    # Xóa các cột phụ trợ
    working = working.drop(columns=['_course_pop', '_user_act'])
    return working

def evaluate_with_noisy_prediction(df, rating_col='Rating', noise_std=1.0, threshold=4.0, seed=42):
    np.random.seed(seed + 1)
    if rating_col not in df.columns:
        raise ValueError('Không tìm thấy cột rating_col trong dataframe')
        
    true = df[rating_col].astype(float).to_numpy()
    pred = np.round(np.clip(true + np.random.normal(0, noise_std, size=len(true)), 1.0, 5.0), 2)
    
    true_rel = (true >= threshold).astype(int)
    pred_rel = (pred >= threshold).astype(int)
    
    p, r, f1, _ = precision_recall_fscore_support(true_rel, pred_rel, average='binary', zero_division=0)
    return p, r, f1

In [22]:
# Cấu hình đường dẫn và tham số (thay thế cho argparse)
input_interactions = r'D:\KLKS\Neo4j\online-course-recommendation-system\mock_user_interactions.csv'
output_interactions = r'D:\KLKS\Neo4j\online-course-recommendation-system\mock_user_interactions_with_ratings.csv'

as_float = False
noise_std = 1.0
threshold = 4.0
force = True # Đặt True để ghi đè nếu cột Rating đã tồn tại
seed = 42

if not os.path.exists(input_interactions):
    print(f'Không tìm thấy file: {input_interactions}')
else:
    # 1. Đọc dữ liệu
    df_interactions = pd.read_csv(input_interactions)
    print(f"Đã tải {len(df_interactions)} dòng dữ liệu tương tác.")

    # 2. Tạo hoặc điền Rating
    if 'Rating' in df_interactions.columns and not force:
        print('Cột Rating đã tồn tại. Chuyển force=True để ghi đè.')
    else:
        df_interactions = generate_ratings_for_sheet(df_interactions, as_float=as_float, seed=seed)
        print(f"Đã điền Rating cho {len(df_interactions)} dòng.")

    # 3. Đánh giá mô phỏng
    p, r, f1 = evaluate_with_noisy_prediction(
        df_interactions, 
        rating_col='Rating', 
        noise_std=noise_std, 
        threshold=threshold, 
        seed=seed
    )
    
    print(f'\n--- Kết quả đánh giá (simulated predictions) với threshold {threshold} ---')
    print(f'Precision: {p:.4f}')
    print(f'Recall:    {r:.4f}')
    print(f'F1-score:  {f1:.4f}\n')

    # 4. Lưu lại thành file CSV mới
    df_interactions.to_csv(output_interactions, index=False)
    print(f'Đã lưu file chứa mock ratings tại: {output_interactions}')
    
    # 5. Hiển thị trực tiếp dưới cell
    display(df_interactions.head(10))

Đã tải 184065 dòng dữ liệu tương tác.
Đã điền Rating cho 184065 dòng.

--- Kết quả đánh giá (simulated predictions) với threshold 4.0 ---
Precision: 0.6556
Recall:    0.5350
F1-score:  0.5892

Đã lưu file chứa mock ratings tại: D:\KLKS\Neo4j\online-course-recommendation-system\mock_user_interactions_with_ratings.csv


,UserID,CourseID,Rating
0,3957,583,3
1,928,444,3
2,6935,660,3
3,6582,637,4
4,4460,1131,3
5,7209,107,3
6,716,619,4
7,9948,587,4
8,758,131,3
9,3656,5,4


In [ ]:
# phân loại danh từ và động từ

import spacy

nlp = spacy.load("en_core_web_sm")

text = """
Become a better manager of people. Develop strategies and skills for hiring, managing performance, and rewarding employees.
"""

doc = nlp(text)

nouns = []
verbs = []

for chunk in doc.noun_chunks:
    tokens = [t.text.lower() for t in chunk if t.pos_ not in ["DET", "PRON"]]
    phrase = " ".join(tokens)
    phrase = phrase.replace(" - ", "-")   # sửa khoảng trắng quanh -
    nouns.append(phrase)

for token in doc:
    if token.pos_ == "VERB":
        verbs.append(token.lemma_.lower())

nouns = sorted(set(nouns))
verbs = sorted(set(verbs))

print("NOUN PHRASES:")
print(nouns)

print("\nVERBS:")
print(verbs)

NOUN PHRASES:
['better manager', 'hiring', 'managing performance', 'people', 'rewarding employees', 'skills', 'strategies']

VERBS:
['become', 'develop', 'manage', 'reward']


In [ ]:
# chuyển vào program type sau khi phân loại
import pandas as pd
import spacy

nlp = spacy.load("en_core_web_sm")

file_path = r"D:\KLKS\Neo4j\online-course-recommendation-system\Online_Courses_Full_2.csv"

# đọc csv với encoding khác
df = pd.read_csv(file_path)

def extract_keywords(text):
    if pd.isna(text):
        return ""

    doc = nlp(str(text))

    nouns = []
    verbs = []

    for chunk in doc.noun_chunks:
        tokens = [t.text.lower() for t in chunk if t.pos_ not in ["DET", "PRON"]]
        phrase = " ".join(tokens)
        phrase = phrase.replace(" - ", "-")
        if phrase:
            nouns.append(phrase)

    for token in doc:
        if token.pos_ == "VERB":
            verbs.append(token.lemma_.lower())

    keywords = sorted(set(nouns + verbs))

    return ", ".join(keywords)

df["Program Type"] = df["Short Intro"].apply(extract_keywords)

df.to_csv(file_path, index=False)

print("Done!")

Done!


In [ ]:
# nhận diện thực thể và lưu file
import pandas as pd
import spacy

# Tải mô hình tiếng Anh của spaCy
nlp = spacy.load("en_core_web_sm")

file_path = r"D:\KLKS\Neo4j\online-course-recommendation-system\Online_Courses_Full_2.csv"

# Đọc file CSV
df = pd.read_csv(file_path)

def extract_ner(text):
    if pd.isna(text):
        return ""

    doc = nlp(str(text))

    entities = []

    # Danh sách các nhãn bạn muốn giữ lại
    target_labels = ["ORG", "PERSON", "PRODUCT", "LANGUAGE", "WORK_OF_ART"]

    for ent in doc.ents:
        if ent.label_ in target_labels:
            entity_text = ent.text.strip()
            if entity_text:
                entities.append(entity_text)

    # Loại bỏ các thực thể trùng lặp và sắp xếp lại
    unique_entities = sorted(set(entities))

    return ", ".join(unique_entities)

# Áp dụng hàm mới để trích xuất NER
df["Program Type"] = df["Short Intro"].apply(extract_ner)

# Lưu lại file CSV
df.to_csv(file_path, index=False,
    encoding="utf-8-sig")

print("Done!")

d:\Python\Python31011\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Done!


In [ ]:
# V + N + NER
import pandas as pd
import spacy
import re

nlp = spacy.load("en_core_web_sm")

file_path = r"D:\KLKS\XuLyDuLieu_ELearning\ELearning_CSV_Tables\KhoaHoc.csv"
df = pd.read_csv(file_path)

def normalize_text(text):
    text = str(text).strip().lower()   # ép toàn bộ về chữ thường
    text = re.sub(r"\s+", " ", text)
    text = text.replace(" - ", "-")
    return text

def unique_preserve_order(items):
    seen = set()
    result = []

    for item in items:
        item = normalize_text(item)
        if not item:
            continue
        if item not in seen:
            seen.add(item)
            result.append(item)

    return result

def extract_program_type(text):
    if pd.isna(text):
        return ""

    doc = nlp(str(text))

    entities = []
    nouns = []
    verbs = []

    target_labels = {"ORG", "PERSON", "PRODUCT", "LANGUAGE", "WORK_OF_ART"}

    for ent in doc.ents:
        if ent.label_ in target_labels:
            entity_text = normalize_text(ent.text)
            if entity_text:
                entities.append(entity_text)

    for chunk in doc.noun_chunks:
        tokens = [t.text.lower() for t in chunk if t.pos_ not in ["DET", "PRON"]]
        phrase = " ".join(tokens)
        phrase = normalize_text(phrase)
        if phrase:
            nouns.append(phrase)

    for token in doc:
        if token.pos_ == "VERB":
            verb = normalize_text(token.lemma_)
            if verb:
                verbs.append(verb)

    combined = entities + nouns + verbs
    unique_items = unique_preserve_order(combined)

    return ", ".join(unique_items)

df["Program Type"] = df["MoTa"].apply(extract_program_type)

df.to_csv(file_path, index=False, encoding="utf-8-sig")

print("Done!")

In [28]:
import pandas as pd

courses = pd.read_csv(r"D:\KLKS\Neo4j\online-course-recommendation-system\Online_Courses_Full.csv", encoding='latin1')
ratings = pd.read_csv(r"D:\KLKS\Neo4j\online-course-recommendation-system\mock_user_interactions.csv")

In [ ]:
# 1. CHECK USER ID MISSING
# =========================
user_ids = sorted(ratings['UserID'].dropna().unique())
expected_user_ids = set(range(1, max(user_ids) + 1))
missing_users = sorted(expected_user_ids - set(user_ids))

print("=== USER ID CHECK ===")
if missing_users:
    print(f"Thiếu {len(missing_users)} userID:", missing_users[:20], "..." if len(missing_users) > 20 else "")
else:
    print("Không thiếu userID nào")

=== USER ID CHECK ===
Không thiếu userID nào


In [ ]:
# 2. CHECK COURSE ID MISSING
# =========================
course_ids = sorted(courses['CourseID'].dropna().unique())
expected_course_ids = set(range(1, max(course_ids) + 1))
missing_courses = sorted(expected_course_ids - set(course_ids))

print("\n=== COURSE ID CHECK ===")
if missing_courses:
    print(f"Thiếu {len(missing_courses)} courseID:", missing_courses[:20], "..." if len(missing_courses) > 20 else "")
else:
    print("Không thiếu courseID nào")


=== COURSE ID CHECK ===
Không thiếu courseID nào


In [26]:
# kiểm tra tb rating trong courses khớp với rating
avg_rating_calc = (
    ratings.groupby('CourseID')['Rating']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'Rating': 'calculated_avg'})
)

# Lấy rating đang lưu trong file courses
courses_rating = courses[['CourseID', 'Rating']].copy()
courses_rating['Rating'] = courses_rating['Rating'].round(2)
courses_rating = courses_rating.rename(columns={'Rating': 'stored_avg'})

# Ghép 2 bảng để so sánh
compare = pd.merge(courses_rating, avg_rating_calc, on='CourseID', how='left')

# Tính độ lệch
compare['diff'] = (compare['stored_avg'] - compare['calculated_avg']).abs()

# Lọc course bị lệch
wrong_courses = compare[compare['diff'] > 0.01]

print("\n=== AVG RATING CHECK ===")
if not wrong_courses.empty:
    print(f"Có {len(wrong_courses)} course bị lệch:")
    print(wrong_courses[['CourseID', 'stored_avg', 'calculated_avg', 'diff']].sort_values(by='CourseID'))
else:
    print("Tất cả course đều đúng average rating")


=== AVG RATING CHECK ===
Tất cả course đều đúng average rating


In [ ]:
# kiểm tra courseID bên rating có hợp lý bên trong courses?
invalid_course_ids = sorted(set(ratings['CourseID']) - set(courses['CourseID']))

print("\n=== INVALID COURSE ID IN RATINGS ===")
if invalid_course_ids:
    print("Có CourseID trong ratings nhưng không tồn tại trong courses:")
    print(invalid_course_ids)
else:
    print("Tất cả CourseID trong ratings đều hợp lệ")


=== INVALID COURSE ID IN RATINGS ===
Tất cả CourseID trong ratings đều hợp lệ


In [31]:
# 8. KHÓA HỌC CHƯA TỪNG ĐƯỢC USER TƯƠNG TÁC
# =========================
ratings_dedup = ratings.drop_duplicates(
    subset=["UserID", "CourseID"],
    keep="last"
)

interacted_course_ids = set(ratings_dedup["CourseID"].unique())

uninteracted_courses = courses[~courses["CourseID"].isin(interacted_course_ids)].copy()
uninteracted_courses = uninteracted_courses.sort_values(by="CourseID")

print("\n=== CÁC KHÓA HỌC CHƯA TỪNG ĐƯỢC USER TƯƠNG TÁC ===")
print(f"Số lượng course chưa được tương tác: {len(uninteracted_courses)}")

if len(uninteracted_courses) > 0:
    print(uninteracted_courses[["CourseID", "Title", "Category"]].to_string(index=False))
else:
    print("Tất cả khóa học đều đã có ít nhất 1 tương tác")

# =========================
# 9. KIỂM TRA COURSEID TRONG RATINGS NHƯNG KHÔNG TỒN TẠI TRONG COURSES
# =========================
invalid_course_ids = sorted(set(ratings_dedup["CourseID"]) - set(courses["CourseID"]))

print("\n=== INVALID COURSE ID IN RATINGS ===")
if invalid_course_ids:
    print("Có CourseID trong ratings nhưng không tồn tại trong courses:")
    print(invalid_course_ids)
else:
    print("Tất cả CourseID trong ratings đều hợp lệ")


=== CÁC KHÓA HỌC CHƯA TỪNG ĐƯỢC USER TƯƠNG TÁC ===
Số lượng course chưa được tương tác: 98
 CourseID                                                                             Title                         Category
       68            Data Engineering, Big Data, and Machine Learning on GCP Specialization                     Data Science
       89             Java Programming and Software Engineering Fundamentals Specialization                 Computer Science
      104                                          Mergers and Acquisitions  Specialization                         Business
      115           Food Sustainability, Mindful Eating, and Healthy Cooking Specialization                           Health
      119                                         Clinical Trials Operations Specialization                           Health
      246                                          Digital Signal Processing Specialization Physical Science and Engineering
      256          Qualitative Re

In [33]:
# CHECK NUMBER OF VIEWERS
# =========================

# 1. Dedup (mỗi user chỉ tính 1 lần / course)
ratings_dedup = ratings.drop_duplicates(
    subset=["UserID", "CourseID"],
    keep="last"
)

# 2. Tính số user unique cho mỗi course
viewer_calc = (
    ratings_dedup.groupby("CourseID")["UserID"]
    .nunique()
    .reset_index()
    .rename(columns={"UserID": "calculated_viewers"})
)

# 3. Lấy số viewers từ courses
course_viewers = courses[["CourseID", "Number of viewers"]].copy()
course_viewers = course_viewers.rename(columns={"Number of viewers": "stored_viewers"})

# 4. Merge để so sánh
compare_viewers = pd.merge(course_viewers, viewer_calc, on="CourseID", how="left")

# Nếu course chưa có ai xem → fill 0
compare_viewers["calculated_viewers"] = compare_viewers["calculated_viewers"].fillna(0).astype(int)

# 5. So sánh lệch
compare_viewers["diff"] = compare_viewers["stored_viewers"] - compare_viewers["calculated_viewers"]

wrong_viewers = compare_viewers[compare_viewers["diff"] != 0]

# 6. In kết quả
print("\n=== NUMBER OF VIEWERS CHECK ===")
if not wrong_viewers.empty:
    print(f"Có {len(wrong_viewers)} course bị lệch viewers:")
    print(
        wrong_viewers[["CourseID", "stored_viewers", "calculated_viewers", "diff"]]
        .sort_values(by="CourseID")
        .to_string(index=False)
    )
else:
    print("Tất cả course đều đúng Number of viewers")


=== NUMBER OF VIEWERS CHECK ===
Có 1 course bị lệch viewers:
 CourseID  stored_viewers  calculated_viewers  diff
       65               0                   1    -1


In [6]:
pip install implicit

     ---------------------------------------- 0.0/693.1 kB ? eta -:--:--
     ----- -------------------------------- 102.4/693.1 kB 2.0 MB/s eta 0:00:01
     -------------------------- ----------- 481.3/693.1 kB 5.0 MB/s eta 0:00:01
     -------------------------------------- 693.1/693.1 kB 5.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
"""
evaluate_content_based.py
=========================
Đánh giá mô hình Content-Based (Neo4j GDS) với các độ đo
giống hệt bên ALS Python:
  - NDCG@5, NDCG@10
  - Precision@5, Precision@10
  - Recall@5, Recall@10
  - Coverage

Chiến lược: Hold-out 20% đánh giá cao nhất mỗi user → test set.
            Phần còn lại dùng làm "hồ sơ" để content-based gợi ý.

Yêu cầu:
    pip install neo4j numpy scikit-learn tqdm
"""

import numpy as np
from neo4j import GraphDatabase
from sklearn.metrics import ndcg_score
from tqdm import tqdm
import random
import warnings
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────────────────────
# CẤU HÌNH KẾT NỐI NEO4J
# ─────────────────────────────────────────────────────────────────────────────
NEO4J_URI      = "neo4j://127.0.0.1:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "12345678"

TOP_K_VALUES   = [5, 10]
TEST_RATIO     = 0.20    # giữ 20% rating của mỗi user làm ground truth
RANDOM_SEED    = 42
MIN_RATINGS    = 5       # bỏ user có ít hơn 5 rating (không đủ để split)


# ─────────────────────────────────────────────────────────────────────────────
# BƯỚC 1: LẤY DỮ LIỆU TỪ NEO4J
# ─────────────────────────────────────────────────────────────────────────────
def fetch_all_ratings(driver):
    """
    Trả về list of dict: {user_id, course_id, rating}
    """
    query = """
        MATCH (nd:NguoiDung)-[dg:DANH_GIA]->(kh:KhoaHoc)
        RETURN nd.id AS user_id, kh.id AS course_id, dg.diem AS rating
        ORDER BY nd.id, dg.diem DESC
    """
    with driver.session() as session:
        result = session.run(query)
        return [{"user_id": r["user_id"],
                 "course_id": r["course_id"],
                 "rating": r["rating"]} for r in result]


def fetch_content_recs_for_user(driver, user_id, train_course_ids, top_k=10):
    """
    Chạy query Content-Based (getProfilePageItems) cho 1 user.
    - train_course_ids: danh sách course user đã học trong train set
      (dùng để loại trừ khỏi kết quả gợi ý)
    """
    query = """
        // Lấy 5 khóa học user thích nhất trong train set làm "profile"
        MATCH (nd:NguoiDung {id: $userId})-[dg:DANH_GIA]->(kh:KhoaHoc)
        WHERE kh.id IN $trainIds
        WITH kh, (dg.diem / 5.0) AS normalizedRating
        ORDER BY normalizedRating DESC
        LIMIT 5

        // Tìm khóa học tương tự qua CONTENT_SIMILAR (đã tính sẵn bởi GDS)
        MATCH (kh)-[rel:CONTENT_SIMILAR]-(q:KhoaHoc)
        WHERE NOT q.id IN $trainIds

        // Đo đám đông + chất lượng
        MATCH (aiDo:NguoiDung)-[dg_q:DANH_GIA]->(q)
        WITH q,
             rel.score          AS contentScore,
             normalizedRating,
             count(dg_q)        AS soLuongDanhGia,
             q.danhGiaTrungBinh AS saoTrungBinh
        WHERE soLuongDanhGia > 0

        // Công thức trọng số (giống controller)
        WITH q,
             soLuongDanhGia,
             saoTrungBinh,
             (contentScore * 0.4)
             + (normalizedRating * 0.2)
             + ((saoTrungBinh / 5.0) * 0.2)
             + (log10(soLuongDanhGia + 1) * 0.2) AS simScore

        // Tránh duplicate nếu q được gợi ý từ nhiều khóa gốc
        WITH q.id AS courseId, max(simScore) AS finalScore
        ORDER BY finalScore DESC
        LIMIT $topK

        RETURN courseId
    """
    with driver.session() as session:
        result = session.run(query,
                             userId=user_id,
                             trainIds=list(train_course_ids),
                             topK=top_k)
        return [r["courseId"] for r in result]


# ─────────────────────────────────────────────────────────────────────────────
# BƯỚC 2: TRAIN / TEST SPLIT  (giống create_train_test_split bên ALS)
# ─────────────────────────────────────────────────────────────────────────────
def create_train_test_split(ratings, test_ratio=TEST_RATIO, min_ratings=MIN_RATINGS):
    """
    Chia theo user: giữ lại 20% rating cuối cùng (sort theo rating desc)
    làm test, phần còn lại là train.

    Returns
    -------
    train : dict  {user_id: set(course_id)}
    test  : dict  {user_id: set(course_id)}
    """
    from collections import defaultdict
    random.seed(RANDOM_SEED)

    user_ratings = defaultdict(list)
    for r in ratings:
        user_ratings[r["user_id"]].append((r["course_id"], r["rating"]))

    train, test = {}, {}
    for uid, items in user_ratings.items():
        if len(items) < min_ratings:
            continue
        # Shuffle để tránh bias
        random.shuffle(items)
        n_test = max(1, int(len(items) * test_ratio))
        test_items  = {cid for cid, _ in items[:n_test]}
        train_items = {cid for cid, _ in items[n_test:]}
        train[uid]  = train_items
        test[uid]   = test_items

    return train, test


# ─────────────────────────────────────────────────────────────────────────────
# BƯỚC 3: METRICS (công thức giống hệt utils.py ALS)
# ─────────────────────────────────────────────────────────────────────────────
def calculate_ndcg_at_k(pred_list, true_set, k, n_items):
    """
    NDCG@K cho 1 user — dùng sklearn.ndcg_score (giống utils.py).
    """
    if not true_set:
        return None

    # true_relevance: binary vector kích thước n_items
    true_relevance = np.zeros(n_items)
    for cid in true_set:
        true_relevance[cid] = 1.0        # cần index; xử lý bên ngoài

    pred_scores = np.zeros(n_items)
    for rank, cid in enumerate(pred_list):
        pred_scores[cid] = len(pred_list) - rank  # score giảm dần theo rank

    try:
        return ndcg_score([true_relevance], [pred_scores], k=k)
    except Exception:
        return None


def evaluate(predictions, test_dict, all_course_ids, k_values=TOP_K_VALUES):
    """
    Tính toàn bộ metrics.

    Parameters
    ----------
    predictions   : dict  {user_id: [list course_id theo thứ tự rank]}
    test_dict     : dict  {user_id: set(course_id)}
    all_course_ids: list  tất cả course_id trong hệ thống
    k_values      : list  [5, 10]

    Returns
    -------
    dict kết quả giống bên ALS utils.evaluate_model()
    """
    # Tạo index liên tục cho NDCG
    course_to_idx = {cid: i for i, cid in enumerate(all_course_ids)}
    n_items = len(all_course_ids)

    results = {}
    all_recommended = set()

    for k in k_values:
        ndcg_list, prec_list, rec_list = [], [], []

        for uid, pred_courses in predictions.items():
            true_courses = test_dict.get(uid, set())
            if not true_courses:
                continue

            pred_k = pred_courses[:k]
            all_recommended.update(pred_courses)

            # --- Precision & Recall (giống utils.py) ---
            hits = len(set(pred_k) & true_courses)
            prec_list.append(hits / k)
            rec_list.append(hits / len(true_courses))

            # --- NDCG (giống calculate_ndcg) ---
            # Map sang index
            pred_idx  = [course_to_idx[c] for c in pred_courses if c in course_to_idx]
            true_idx  = {course_to_idx[c] for c in true_courses  if c in course_to_idx}

            true_rel  = np.zeros(n_items)
            for idx in true_idx:
                true_rel[idx] = 1.0

            pred_sc = np.zeros(n_items)
            for rank, idx in enumerate(pred_idx):
                pred_sc[idx] = len(pred_idx) - rank

            try:
                score = ndcg_score([true_rel], [pred_sc], k=k)
                ndcg_list.append(score)
            except Exception:
                pass

        results[f"ndcg@{k}"]      = float(np.mean(ndcg_list))      if ndcg_list  else 0.0
        results[f"precision@{k}"] = float(np.mean(prec_list))      if prec_list  else 0.0
        results[f"recall@{k}"]    = float(np.mean(rec_list))        if rec_list   else 0.0

    # Coverage
    results["coverage"] = len(all_recommended) / n_items if n_items > 0 else 0.0

    return results


# ─────────────────────────────────────────────────────────────────────────────
# BƯỚC 4: RANDOM BASELINE  (giống compare_to_random_baseline bên ALS)
# ─────────────────────────────────────────────────────────────────────────────
def random_baseline(test_dict, all_course_ids, k=10, n_samples=500):
    """Tính NDCG@K ngẫu nhiên để làm mốc so sánh."""
    random.seed(RANDOM_SEED)
    sampled_users = random.sample(list(test_dict.keys()),
                                  min(n_samples, len(test_dict)))

    random_preds = {
        uid: random.sample(all_course_ids, min(k, len(all_course_ids)))
        for uid in sampled_users
    }
    test_sampled = {uid: test_dict[uid] for uid in sampled_users}

    res = evaluate(random_preds, test_sampled, all_course_ids, k_values=[k])
    return res


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print("=" * 65)
    print("  CONTENT-BASED EVALUATION  (Neo4j GDS — same metrics as ALS)")
    print("=" * 65)

    driver = GraphDatabase.driver(NEO4J_URI,
                                  auth=(NEO4J_USER, NEO4J_PASSWORD))

    # ── 1. Lấy toàn bộ ratings ──────────────────────────────────────────────
    print("\n[1/5] Fetching ratings from Neo4j...")
    ratings = fetch_all_ratings(driver)
    print(f"      Total ratings: {len(ratings):,}")

    all_course_ids = list({r["course_id"] for r in ratings})
    print(f"      Unique courses: {len(all_course_ids):,}")

    # ── 2. Train / Test split ────────────────────────────────────────────────
    print(f"\n[2/5] Creating {int((1-TEST_RATIO)*100)}/{int(TEST_RATIO*100)} train/test split...")
    train_dict, test_dict = create_train_test_split(ratings)
    print(f"      Users in split: {len(test_dict):,}")

    # ── 3. Thu thập gợi ý từ Neo4j ──────────────────────────────────────────
    print(f"\n[3/5] Fetching content-based recommendations from Neo4j...")
    print(f"      (Running getProfilePageItems query for each user)")

    predictions = {}
    max_k = max(TOP_K_VALUES)

    for uid in tqdm(list(test_dict.keys()), desc="  Users"):
        train_ids = train_dict.get(uid, set())
        if not train_ids:
            continue
        recs = fetch_content_recs_for_user(driver, uid, train_ids, top_k=max_k)
        if recs:
            predictions[uid] = recs

    print(f"      Users with recommendations: {len(predictions):,} / {len(test_dict):,}")

    # ── 4. Tính metrics ──────────────────────────────────────────────────────
    print(f"\n[4/5] Calculating metrics...")
    results = evaluate(predictions, test_dict, all_course_ids)

    # ── 5. Random baseline ───────────────────────────────────────────────────
    print(f"\n[5/5] Calculating random baseline (sample 500 users)...")
    baseline = random_baseline(test_dict, all_course_ids, k=10)

    driver.close()

    # ─────────────────────────────────────────────────────────────────────────
    # IN KẾT QUẢ
    # ─────────────────────────────────────────────────────────────────────────
    print("\n" + "=" * 65)
    print("  RESULTS — Content-Based Model")
    print("=" * 65)

    metrics_order = [
        "ndcg@5", "ndcg@10",
        "precision@5", "precision@10",
        "recall@5", "recall@10",
        "coverage"
    ]

    for m in metrics_order:
        v = results.get(m, 0.0)
        print(f"  {m:<18s}: {v:.4f}")

    print()
    print(f"  {'Random NDCG@10':<18s}: {baseline.get('ndcg@10', 0):.4f}")
    rand_ndcg = baseline.get("ndcg@10", 1e-9)
    cb_ndcg   = results.get("ndcg@10",  0)
    if rand_ndcg > 0:
        improvement = cb_ndcg / rand_ndcg
        print(f"  {'Improvement':<18s}: {improvement:.2f}x vs random")

    print("\n" + "=" * 65)
    print("  INTERPRETATION")
    print("=" * 65)
    ndcg10 = results.get("ndcg@10", 0)
    if ndcg10 > 0.30:
        print("  ✅ Excellent — model is production-ready.")
    elif ndcg10 > 0.15:
        print("  ✅ Good — model shows meaningful signal.")
    elif ndcg10 > 0.05:
        print("  ⚠️  Moderate — consider tuning similarityCutoff or weights.")
    else:
        print("  ❌ Low — check CONTENT_SIMILAR edges exist (run GDS first).")

    print("\n  TIP: Compare these numbers with your ALS results file")
    print("       (results/als_results.pkl) for apples-to-apples comparison.")
    print("=" * 65)

    return results


if __name__ == "__main__":
    main()

  CONTENT-BASED EVALUATION  (Neo4j GDS — same metrics as ALS)

[1/5] Fetching ratings from Neo4j...
      Total ratings: 135,250
      Unique courses: 1,534

[2/5] Creating 80/20 train/test split...
      Users in split: 9,576

[3/5] Fetching content-based recommendations from Neo4j...
      (Running getProfilePageItems query for each user)


  Users: 100%|██████████| 9576/9576 [00:47<00:00, 203.74it/s]


      Users with recommendations: 9,555 / 9,576

[4/5] Calculating metrics...

[5/5] Calculating random baseline (sample 500 users)...

  RESULTS — Content-Based Model
  ndcg@5            : 0.0059
  ndcg@10           : 0.0062
  precision@5       : 0.0028
  precision@10      : 0.0015
  recall@5          : 0.0052
  recall@10         : 0.0058
  coverage          : 0.7066

  Random NDCG@10    : 0.0046
  Improvement       : 1.33x vs random

  INTERPRETATION
  ❌ Low — check CONTENT_SIMILAR edges exist (run GDS first).

  TIP: Compare these numbers with your ALS results file
       (results/als_results.pkl) for apples-to-apples comparison.


In [7]:
"""
Instacart Recommender System - ALS Model (Tutorial 2)

This module implements Alternating Least Squares (ALS) matrix factorization
for collaborative filtering using the implicit library.
a
ALS is a classical approach that learns latent factors for users and items
by decomposing the user-item interaction matrix. It works well with implicit
feedback data (purchases, clicks) where we only observe positive interactions.

Key advantages:
- Fast training (can use GPU)
- Scalable to millions of users/items
- Produces interpretable embeddings
- Proven technique used in production systems

The model learns:
- User embeddings: each user → 64-dim vector
- Item embeddings: each product → 64-dim vector
- Recommendation score = dot product of user and item vectors
"""

import numpy as np
import scipy.sparse as sp
from implicit.als import AlternatingLeastSquares
from implicit.nearest_neighbours import bm25_weight
import warnings
warnings.filterwarnings('ignore')


def train_als(train_matrix, factors=64, regularization=0.01, iterations=15, 
              use_gpu=True, alpha=40):
    """
    Train ALS model on user-item interaction matrix.
    
    The model learns latent factors that capture user preferences and product
    characteristics. Users with similar purchase histories get similar embeddings,
    and products that are frequently bought together get similar embeddings.
    
    Parameters:
    -----------
    train_matrix : scipy sparse matrix (users × items)
        Training interaction matrix. Non-zero entries indicate purchases.
        Values can be purchase counts (implicit confidence) or binary (0/1).
    
    factors : int, default=64
        Dimensionality of the latent factors (embedding size).
        Higher = more expressive but slower and may overfit.
        Common values: 32, 64, 128
    
    regularization : float, default=0.01
        L2 regularization strength to prevent overfitting.
        Higher values = more regularization = simpler model.
        Typical range: 0.001 to 0.1
    
    iterations : int, default=15
        Number of ALS iterations.
        Each iteration alternates between updating user and item factors.
        Usually converges in 10-20 iterations.
    
    use_gpu : bool, default=True
        Use GPU acceleration if available (requires cupy).
        Falls back to CPU if GPU not available.
    
    alpha : float, default=40
        Confidence scaling for implicit feedback.
        Transforms raw counts to confidence: confidence = 1 + alpha * count
        Higher alpha = more weight on observed interactions.
    
    Returns:
    --------
    model : AlternatingLeastSquares
        Trained ALS model with learned user and item factors
    
    Notes:
    ------
    The loss function being minimized is:
    L = Σ c_ui * (p_ui - user_i · item_j)² + λ(||user_i||² + ||item_j||²)
    
    where:
    - c_ui is confidence (1 + alpha * r_ui for observed, 1 for unobserved)
    - p_ui is preference (1 for observed, 0 for unobserved)
    - λ is regularization
    """
    
    print("\n" + "="*70)
    print("TRAINING ALS MODEL")
    print("="*70)
    
    print(f"\nModel Configuration:")
    print(f"  Embedding dimension: {factors}")
    print(f"  Regularization: {regularization}")
    print(f"  Iterations: {iterations}")
    print(f"  Alpha (confidence): {alpha}")
    print(f"  Use GPU: {use_gpu}")
    
    print(f"\nTraining Data:")
    print(f"  Shape: {train_matrix.shape[0]:,} users × {train_matrix.shape[1]:,} items")
    print(f"  Non-zero entries: {train_matrix.nnz:,}")
    print(f"  Sparsity: {(1 - train_matrix.nnz / (train_matrix.shape[0] * train_matrix.shape[1])):.2%}")
    
    # Apply BM25 weighting to reduce popularity bias
    # This down-weights popular items so they don't dominate recommendations
    print("\nApplying BM25 weighting to reduce popularity bias...")
    train_matrix_weighted = bm25_weight(train_matrix, K1=100, B=0.8)
    
    # Convert to CSR format (users × items)
    # Note: implicit library works with both formats, we use users × items
    train_matrix_csr = train_matrix_weighted.tocsr()
    
    # Initialize model
    print("\nInitializing ALS model...")
    model = AlternatingLeastSquares(
        factors=factors,  #64
        regularization=regularization,  #0.01
        iterations=iterations,  #15
        use_gpu=use_gpu,  #Currently False
        random_state=42
    )
    
    # Train model
    print("\nTraining (this may take 1-2 minutes)...")
    model.fit(train_matrix_csr, show_progress=True)
    
    print("\n" + "="*70)
    print("TRAINING COMPLETE")
    print("="*70)
    
    # Model now contains:
    # - model.user_factors: (n_users × factors) matrix
    # - model.item_factors: (n_items × factors) matrix
    
    return model


def predict_als(model, train_matrix, user_ids=None, k=10, 
                filter_already_purchased=True):
    """
    Generate top-K recommendations for users using trained ALS model.
    
    For each user, computes scores for all items using dot product of
    user embedding and item embeddings. Returns top-K items with highest scores.
    
    Parameters:
    -----------
    model : AlternatingLeastSquares
        Trained ALS model
    
    train_matrix : scipy sparse matrix (users × items)
        Training data used to filter out already purchased items
    
    user_ids : list or None
        User indices to generate recommendations for.
        If None, generates for all users.
    
    k : int, default=10
        Number of recommendations per user
    
    filter_already_purchased : bool, default=True
        If True, exclude items the user already purchased in training.
        Typically True for evaluation, False for actual deployment.
    
    Returns:
    --------
    predictions : dict
        {user_idx: [list of k product indices]}
        Products are ordered by predicted score (highest first)
    
    scores : dict
        {user_idx: [list of k scores]}
        Corresponding confidence scores for each recommendation
    
    Notes:
    ------
    Recommendation score for user u and item i:
    score(u, i) = user_embedding[u] · item_embedding[i]
    
    Higher score = more likely to purchase
    """
    
    print("\nGenerating recommendations...")
    
    # Default to all users
    if user_ids is None:
        user_ids = list(range(train_matrix.shape[0]))
    
    predictions = {}
    scores_dict = {}
    
    # Convert to CSR format (users × items)
    train_matrix_csr = train_matrix.tocsr()
    
    print(f"  Predicting for {len(user_ids):,} users...")
    
    # Use batch recommendation for efficiency
    # recommend returns (item_ids, scores) for batch of users
    batch_items, batch_scores = model.recommend(
        userid=user_ids,
        user_items=train_matrix_csr,
        N=k,
        filter_already_liked_items=filter_already_purchased
    )
    
    # Convert batch results to dictionary format
    for i, user_idx in enumerate(user_ids):
        predictions[user_idx] = batch_items[i].tolist()
        scores_dict[user_idx] = batch_scores[i].tolist()
    
    print(f"  Generated {k} recommendations per user")
    
    return predictions, scores_dict


def get_similar_items(model, item_ids, n=10):
    """
    Find items similar to given items.
    
    Uses item embeddings to find products that are frequently purchased
    together with the query items. Useful for "customers also bought"
    style recommendations.
    
    Parameters:
    -----------
    model : AlternatingLeastSquares
        Trained ALS model
    
    item_ids : list
        Product indices to find similar items for
    
    n : int, default=10
        Number of similar items to return per query item
    
    Returns:
    --------
    dict : {item_id: [(similar_item_id, score), ...]}
    """
    
    similar_items = {}
    
    for item_id in item_ids:
        # Find similar items using cosine similarity of embeddings
        items, scores = model.similar_items(itemid=item_id, N=n+1)
        
        # Remove the query item itself (first result)
        similar_items[item_id] = list(zip(items[1:], scores[1:]))
    
    return similar_items


def get_item_embeddings(model):
    """
    Extract item embedding matrix from trained model.
    
    These embeddings capture product characteristics learned from purchase patterns.
    Similar products (e.g., different brands of milk) have similar embeddings.
    
    Parameters:
    -----------
    model : AlternatingLeastSquares
        Trained ALS model
    
    Returns:
    --------
    embeddings : numpy array, shape (n_items, factors)
        Item factor matrix where each row is a product's embedding vector
    
    Notes:
    ------
    Can be used for:
    - Visualization (PCA/t-SNE to 2D)
    - Item-to-item similarity
    - Product clustering
    - Cold start for new products (use content features to predict embedding)
    """
    
    return model.item_factors


def get_user_embeddings(model):
    """
    Extract user embedding matrix from trained model.
    
    These embeddings capture user preferences learned from purchase history.
    Users with similar tastes have similar embeddings.
    
    Parameters:
    -----------
    model : AlternatingLeastSquares
        Trained ALS model
    
    Returns:
    --------
    embeddings : numpy array, shape (n_users, factors)
        User factor matrix where each row is a user's embedding vector
    
    Notes:
    ------
    Can be used for:
    - User segmentation
    - Finding similar users
    - Cold start analysis (understand what makes a user predictable)
    """
    
    return model.user_factors


def explain_recommendations(model, user_idx, item_idx, train_matrix, 
                           product_info, top_n=5):
    """
    Explain why an item was recommended to a user.
    
    Shows which products the user previously purchased that led to
    this recommendation (based on embedding similarity).
    
    Parameters:
    -----------
    model : AlternatingLeastSquares
        Trained ALS model
    
    user_idx : int
        User index
    
    item_idx : int
        Recommended item index
    
    train_matrix : scipy sparse matrix
        Training data showing user's purchase history
    
    product_info : DataFrame
        Product metadata
    
    top_n : int
        Number of explanatory items to return
    
    Returns:
    --------
    list : [(product_name, contribution_score), ...]
    
    Notes:
    ------
    Contribution score = user_history_item · recommended_item
    Items with highest contribution most influenced this recommendation.
    """
    
    # Get user's purchase history
    user_items = train_matrix[user_idx].nonzero()[1]
    
    if len(user_items) == 0:
        return []
    
    # Get embeddings
    user_embedding = model.user_factors[user_idx]
    item_embedding = model.item_factors[item_idx]
    history_embeddings = model.item_factors[user_items]
    
    # Calculate contribution of each history item
    contributions = history_embeddings.dot(item_embedding)
    
    # Get top contributing items
    top_indices = np.argsort(contributions)[::-1][:top_n]
    
    explanations = []
    for idx in top_indices:
        history_item_idx = user_items[idx]
        contrib_score = contributions[idx]
        
        # Get product name
        prod_row = product_info[product_info['product_idx'] == history_item_idx]
        if not prod_row.empty:
            product_name = prod_row['product_name'].values[0]
            explanations.append((product_name, contrib_score))
    
    return explanations


def get_model_statistics(model, train_matrix):
    """
    Calculate statistics about the trained model.
    
    Useful for understanding model complexity and potential issues.
    
    Parameters:
    -----------
    model : AlternatingLeastSquares
        Trained ALS model
    
    train_matrix : scipy sparse matrix
        Training data
    
    Returns:
    --------
    dict : Model statistics
    """
    
    stats = {}
    
    # Embedding statistics
    user_factors = model.user_factors
    item_factors = model.item_factors
    
    stats['n_users'] = user_factors.shape[0]
    stats['n_items'] = item_factors.shape[0]
    stats['embedding_dim'] = user_factors.shape[1]
    
    # Check for potential issues
    stats['user_embedding_norm_mean'] = np.linalg.norm(user_factors, axis=1).mean()
    stats['item_embedding_norm_mean'] = np.linalg.norm(item_factors, axis=1).mean()
    
    # Sparsity
    stats['train_sparsity'] = 1 - (train_matrix.nnz / (train_matrix.shape[0] * train_matrix.shape[1]))
    
    # Average interactions per user
    stats['avg_items_per_user'] = train_matrix.nnz / train_matrix.shape[0]
    
    return stats


def save_als_model(model, filepath):
    """
    Save trained ALS model to disk.
    
    Parameters:
    -----------
    model : AlternatingLeastSquares
        Trained model
    
    filepath : str
        Path to save model (will create .npz file)
    """
    
    import pickle
    
    # Save model using pickle
    with open(filepath, 'wb') as f:
        pickle.dump(model, f)
    
    print(f"\nALS model saved to: {filepath}")


def load_als_model(filepath):
    """
    Load saved ALS model from disk.
    
    Parameters:
    -----------
    filepath : str
        Path to saved model file
    
    Returns:
    --------
    model : AlternatingLeastSquares
        Loaded model
    """
    
    import pickle
    
    with open(filepath, 'rb') as f:
        model = pickle.load(f)
    
    print(f"\nALS model loaded from: {filepath}")
    
    return model


# =============================================================================
# 
# =============================================================================

if __name__ == "__main__":
    """
    Example showing how to train and use the ALS model.
    This is just for testing - actual usage is in run_tutorial_2.py
    """
    
    print("ALS Model Module - Example Usage")
    print("="*70)
    print("\nThis module contains functions for training ALS models.")
    print("For full tutorial, run: python run_tutorial_2.py")
    print("\nKey functions:")
    print("  - train_als(): Train the model")
    print("  - predict_als(): Generate recommendations")
    print("  - get_item_embeddings(): Extract embeddings for visualization")
    print("  - explain_recommendations(): Understand why items were recommended")

ALS Model Module - Example Usage

This module contains functions for training ALS models.
For full tutorial, run: python run_tutorial_2.py

Key functions:
  - train_als(): Train the model
  - predict_als(): Generate recommendations
  - get_item_embeddings(): Extract embeddings for visualization
  - explain_recommendations(): Understand why items were recommended


In [14]:
import pickle
import scipy.sparse as sp
from neo4j import GraphDatabase

NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678"

def load_pkl(filepath):
    with open(filepath, "rb") as f:
        return pickle.load(f)

def push_recommendations_to_neo4j(predictions_dict, scores_dict, user_map_raw, item_map_raw):
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    
    print("Đang xóa dữ liệu ALS cũ trong DB...")
    with driver.session() as session:
        session.run("MATCH ()-[r:ALS_RECOMMEND]->() DELETE r")
        
    print("Đang trích xuất và chuẩn hóa bản đồ ánh xạ ID...")
    
    # Lấy trực tiếp từ key đã debug và ép kiểu về int thuần của Python để xóa bỏ np.int64
    idx_to_user = {int(k): int(v) for k, v in user_map_raw['idx_to_user'].items()}
    idx_to_item = {int(k): int(v) for k, v in item_map_raw['idx_to_product'].items()}
    
    query = """
    UNWIND $batch AS row
    MATCH (u:NguoiDung {id: row.userId})
    MATCH (k:KhoaHoc {id: row.courseId})
    MERGE (u)-[r:ALS_RECOMMEND]->(k)
    SET r.score = row.score
    """
    
    with driver.session() as session:
        batch_data = []
        
        for user_idx, course_indices in predictions_dict.items():
            # Chuyển đổi key sang int để khớp với idx_to_user
            u_idx_int = int(user_idx)
            if u_idx_int not in idx_to_user:
                continue
                
            real_user_id = idx_to_user[u_idx_int] 
            
            for i, course_idx in enumerate(course_indices):
                c_idx_int = int(course_idx)
                if c_idx_int not in idx_to_item:
                    continue
                    
                real_course_id = idx_to_item[c_idx_int]
                score = scores_dict[user_idx][i]
                
                batch_data.append({
                    "userId": real_user_id,
                    "courseId": real_course_id,
                    "score": float(score)
                })
        
        print(f"Đang đẩy {len(batch_data)} bản ghi gợi ý lên Neo4j...")
        if len(batch_data) > 0:
            session.run(query, batch=batch_data)
            print("Hoàn tất đẩy dữ liệu vào Neo4j!")
        else:
            print("Cảnh báo: Không có bản ghi nào hợp lệ để đẩy. Vui lòng kiểm tra lại ID node trong Neo4j.")
        
    driver.close()

if __name__ == "__main__":
    
    print("Đang load model, matrix và mapping files...")
    model = load_pkl(r"D:\KLKS\Recommendation\als_model.pkl")
    train_matrix = sp.load_npz(r"D:\KLKS\processed\train_matrix.npz")
    
    user_map = load_pkl(r"D:\KLKS\processed\user_mapping.pkl") 
    item_map = load_pkl(r"D:\KLKS\processed\item_mapping.pkl")
    
    # Sinh dữ liệu dự đoán
    predictions, scores = predict_als(model, train_matrix, k=10)
    
    # Đẩy lên cơ sở dữ liệu
    push_recommendations_to_neo4j(predictions, scores, user_map, item_map)

Đang load model, matrix và mapping files...

Generating recommendations...
  Predicting for 9,424 users...
  Generated 10 recommendations per user
Đang xóa dữ liệu ALS cũ trong DB...
Đang trích xuất và chuẩn hóa bản đồ ánh xạ ID...
Đang đẩy 94240 bản ghi gợi ý lên Neo4j...
Hoàn tất đẩy dữ liệu vào Neo4j!


In [12]:
import pickle

def debug_mapping(filepath):
    print(f"\n--- Đang kiểm tra file: {filepath} ---")
    with open(filepath, "rb") as f:
        data = pickle.load(f)
    
    print(f"1. Kiểu dữ liệu (Type): {type(data)}")
    
    if isinstance(data, dict):
        print("2. Các Key cấp 1:")
        for k in list(data.keys())[:5]: # Chỉ in 5 key đầu tiên
            print(f"   - Key: '{k}' (Kiểu: {type(k)})")
            
            # Nếu giá trị bên trong cũng là dict, in thử vài phần tử
            val = data[k]
            if isinstance(val, dict):
                print(f"     => Giá trị là Dictionary chứa {len(val)} phần tử.")
                print(f"     => Ví dụ vài phần tử bên trong: {list(val.items())[:3]}")
            else:
                print(f"     => Giá trị (Value): {val} (Kiểu: {type(val)})")
    else:
        print("Dữ liệu không phải là Dictionary.")

# Thay đổi đường dẫn cho đúng với máy của bạn
debug_mapping(r"D:\KLKS\processed\user_mapping.pkl")
debug_mapping(r"D:\KLKS\processed\item_mapping.pkl")


--- Đang kiểm tra file: D:\KLKS\processed\user_mapping.pkl ---
1. Kiểu dữ liệu (Type): <class 'dict'>
2. Các Key cấp 1:
   - Key: 'user_to_idx' (Kiểu: <class 'str'>)
     => Giá trị là Dictionary chứa 9424 phần tử.
     => Ví dụ vài phần tử bên trong: [(np.int64(90), 0), (np.int64(107), 1), (np.int64(112), 2)]
   - Key: 'idx_to_user' (Kiểu: <class 'str'>)
     => Giá trị là Dictionary chứa 9424 phần tử.
     => Ví dụ vài phần tử bên trong: [(0, np.int64(90)), (1, np.int64(107)), (2, np.int64(112))]

--- Đang kiểm tra file: D:\KLKS\processed\item_mapping.pkl ---
1. Kiểu dữ liệu (Type): <class 'dict'>
2. Các Key cấp 1:
   - Key: 'product_to_idx' (Kiểu: <class 'str'>)
     => Giá trị là Dictionary chứa 1500 phần tử.
     => Ví dụ vài phần tử bên trong: [(np.int64(1), 0), (np.int64(2), 1), (np.int64(3), 2)]
   - Key: 'idx_to_product' (Kiểu: <class 'str'>)
     => Giá trị là Dictionary chứa 1500 phần tử.
     => Ví dụ vài phần tử bên trong: [(0, np.int64(1)), (1, np.int64(2)), (2, np.int64

In [1]:
import numpy as np
from neo4j import GraphDatabase
from sklearn.metrics import ndcg_score
from collections import defaultdict
from tqdm import tqdm
import random
import warnings

warnings.filterwarnings("ignore")

# CẤU HÌNH NEO4J
NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678" # Đổi lại cho đúng với Neo4j của bạn

# CẤU HÌNH ĐÁNH GIÁ (Giống ALS)
TEST_RATIO = 0.2
MIN_RATINGS = 5
TOP_K_VALUES = [5, 10]
RANDOM_SEED = 42

def fetch_all_ratings(driver):
    """Lấy toàn bộ ĐÁNH_GIÁ từ Neo4j để chia Train/Test"""
    query = """
        MATCH (nd:NguoiDung)-[dg:DANH_GIA]->(kh:KhoaHoc)
        RETURN nd.id AS user_id, kh.id AS course_id, dg.diem AS rating
    """
    with driver.session() as session:
        result = session.run(query)
        return [{"user_id": r["user_id"], "course_id": r["course_id"], "rating": r["rating"]} for r in result]

def create_train_test_split(ratings):
    """Chia tập train/test theo tỷ lệ 80/20 cho mỗi user giống hàm create_train_test_split của ALS"""
    random.seed(RANDOM_SEED)
    user_ratings = defaultdict(list)
    for r in ratings:
        user_ratings[r["user_id"]].append((r["course_id"], r["rating"]))

    train, test = {}, {}
    for uid, items in user_ratings.items():
        if len(items) < MIN_RATINGS:
            continue
            
        random.shuffle(items)
        n_test = max(1, int(len(items) * TEST_RATIO))
        
        test[uid] = {cid for cid, _ in items[:n_test]}
        train[uid] = {cid for cid, _ in items[n_test:]}

    return train, test

def fetch_recommendations_from_neo4j(driver, user_id, train_course_ids, model_type="HYBRID", top_k=10):
    """
    Chạy truy vấn Cypher tương ứng với API trong RecommendationController.cs
    Nhưng RÀNG BUỘC lịch sử user bằng $trainIds để đánh giá khách quan.
    """
    if model_type == "HYBRID":
        # Logic giống hàm GetUserProfileBasedRecommendations (Hybrid Content + Popularity + Profile)
        query = """
            // 1. Chỉ dùng các khóa học trong Train Set để tạo Profile
            MATCH (nd:NguoiDung {id: $userId})-[dg:DANH_GIA]->(kh:KhoaHoc)
            WHERE kh.id IN $trainIds
            WITH kh, (dg.diem / 5.0) AS normalizedRating
            ORDER BY normalizedRating DESC
            LIMIT 5

            // 2. Tìm khóa học tương tự về nội dung
            MATCH (kh)-[rel:CONTENT_SIMILAR]-(q:KhoaHoc)
            WHERE NOT q.id IN $trainIds

            // 3. Đo độ phổ biến (cho phép lấy tổng thể để giống C#)
            OPTIONAL MATCH (aiDo:NguoiDung)-[dg_q:DANH_GIA]->(q)
            WITH q, rel.score AS contentScore, normalizedRating, 
                 count(dg_q) AS soLuongDanhGia, coalesce(q.danhGiaTrungBinh, 0.0) AS saoTrungBinh
            WHERE soLuongDanhGia > 0

            // 4. Tính toán trọng số y hệt Controller C#
            WITH q, soLuongDanhGia, saoTrungBinh,
                 (contentScore * 0.4) + (normalizedRating * 0.2) + ((saoTrungBinh / 5.0) * 0.2) + (log10(soLuongDanhGia + 1) * 0.2) AS simScore

            WITH q.id AS courseId, max(simScore) AS finalScore
            ORDER BY finalScore DESC
            LIMIT $topK
            RETURN courseId
        """
    elif model_type == "PURE_CONTENT":
        # Thuần Content-Based (Chỉ dựa vào mức độ tương đồng nội dung CONTENT_SIMILAR)
        query = """
            MATCH (nd:NguoiDung {id: $userId})-[dg:DANH_GIA]->(kh:KhoaHoc)
            WHERE kh.id IN $trainIds AND dg.diem >= 3.0
            WITH kh
            MATCH (kh)-[rel:CONTENT_SIMILAR]-(q:KhoaHoc)
            WHERE NOT q.id IN $trainIds
            WITH q.id AS courseId, max(rel.score) AS finalScore
            ORDER BY finalScore DESC
            LIMIT $topK
            RETURN courseId
        """

    with driver.session() as session:
        result = session.run(query, userId=user_id, trainIds=list(train_course_ids), topK=top_k)
        return [r["courseId"] for r in result]

def evaluate(predictions, test_dict, all_course_ids, k_values=TOP_K_VALUES):
    """Tính các metric y hệt evaluate_model() trong ALS utils"""
    course_to_idx = {cid: i for i, cid in enumerate(all_course_ids)}
    n_items = len(all_course_ids)
    results = {}
    all_recommended = set()

    for k in k_values:
        ndcg_list, prec_list, rec_list = [], [], []

        for uid, pred_courses in predictions.items():
            true_courses = test_dict.get(uid, set())
            if not true_courses:
                continue

            pred_k = pred_courses[:k]
            all_recommended.update(pred_courses)

            # Precision & Recall
            hits = len(set(pred_k) & true_courses)
            prec_list.append(hits / k if k > 0 else 0)
            rec_list.append(hits / len(true_courses) if len(true_courses) > 0 else 0)

            # NDCG@K
            pred_idx = [course_to_idx[c] for c in pred_courses if c in course_to_idx]
            true_idx = {course_to_idx[c] for c in true_courses if c in course_to_idx}

            true_rel = np.zeros(n_items)
            for idx in true_idx: true_rel[idx] = 1.0

            pred_sc = np.zeros(n_items)
            for rank, idx in enumerate(pred_idx):
                pred_sc[idx] = len(pred_idx) - rank

            try:
                if len(pred_idx) > 0 and len(true_idx) > 0:
                    score = ndcg_score([true_rel], [pred_sc], k=k)
                    ndcg_list.append(score)
            except Exception:
                pass

        results[f"ndcg@{k}"] = float(np.mean(ndcg_list)) if ndcg_list else 0.0
        results[f"precision@{k}"] = float(np.mean(prec_list)) if prec_list else 0.0
        results[f"recall@{k}"] = float(np.mean(rec_list)) if rec_list else 0.0

    results["coverage"] = len(all_recommended) / n_items if n_items > 0 else 0.0
    return results

def main():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    
    print("\n1. Đang tải dữ liệu từ Neo4j...")
    ratings = fetch_all_ratings(driver)
    all_course_ids = list({r["course_id"] for r in ratings})
    
    print("2. Đang tạo Train/Test Split (80/20)...")
    train_dict, test_dict = create_train_test_split(ratings)
    
    # Để code chạy nhanh khi đánh giá, ta lấy ngẫu nhiên 500-1000 user để test
    sample_users = random.sample(list(test_dict.keys()), min(1000, len(test_dict)))
    
    predictions_hybrid = {}
    predictions_content = {}
    
    print(f"\n3. Tiến hành tạo gợi ý qua Cypher cho {len(sample_users)} Users...")
    for uid in tqdm(sample_users):
        train_ids = train_dict.get(uid, set())
        if not train_ids: continue
            
        # Chạy model Hybrid (như trong C# user-profile)
        predictions_hybrid[uid] = fetch_recommendations_from_neo4j(
            driver, uid, train_ids, model_type="HYBRID", top_k=10
        )
        
        # Chạy model Content thuần túy
        predictions_content[uid] = fetch_recommendations_from_neo4j(
            driver, uid, train_ids, model_type="PURE_CONTENT", top_k=10
        )

    driver.close()

    print("\n4. ĐÁNH GIÁ MÔ HÌNH VÀ SO SÁNH...")
    res_hybrid = evaluate(predictions_hybrid, test_dict, all_course_ids)
    res_content = evaluate(predictions_content, test_dict, all_course_ids)

    print("\n" + "="*60)
    print(f"{'Metric':<15} | {'Pure Content':<18} | {'Hybrid (Profile)':<18}")
    print("="*60)
    
    metrics = ["ndcg@5", "ndcg@10", "precision@5", "precision@10", "recall@5", "recall@10", "coverage"]
    for m in metrics:
        val_content = res_content.get(m, 0.0)
        val_hybrid = res_hybrid.get(m, 0.0)
        print(f"{m:<15} | {val_content:<18.4f} | {val_hybrid:<18.4f}")
    
    print("="*60)
    print("\n* So sánh các hệ số này với hệ số của ALS bạn đã lưu trong 'als_results.pkl'")

if __name__ == "__main__":
    main()


1. Đang tải dữ liệu từ Neo4j...
2. Đang tạo Train/Test Split (80/20)...

3. Tiến hành tạo gợi ý qua Cypher cho 1000 Users...


100%|██████████| 1000/1000 [00:11<00:00, 84.58it/s]



4. ĐÁNH GIÁ MÔ HÌNH VÀ SO SÁNH...

Metric          | Pure Content       | Hybrid (Profile)  
ndcg@5          | 0.0007             | 0.0058            
ndcg@10         | 0.0023             | 0.0059            
precision@5     | 0.0004             | 0.0028            
precision@10    | 0.0011             | 0.0014            
recall@5        | 0.0011             | 0.0055            
recall@10       | 0.0048             | 0.0055            
coverage        | 0.6141             | 0.5567            

* So sánh các hệ số này với hệ số của ALS bạn đã lưu trong 'als_results.pkl'


In [ ]:
import numpy as np
from neo4j import GraphDatabase
from sklearn.metrics import ndcg_score
from collections import defaultdict
from tqdm import tqdm
import random
import warnings

warnings.filterwarnings("ignore")

# CẤU HÌNH NEO4J
NEO4J_URI = "neo4j://127.0.0.1:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "12345678" # Đổi lại cho đúng với Neo4j của bạn

# CẤU HÌNH ĐÁNH GIÁ
TEST_RATIO = 0.2
MIN_RATINGS = 5
TOP_K_VALUES = [5, 10]
RANDOM_SEED = 42

def fetch_all_ratings(driver):
    """Lấy toàn bộ ĐÁNH_GIÁ từ Neo4j để chia Train/Test"""
    query = """
        MATCH (nd:NguoiDung)-[dg:DANH_GIA]->(kh:KhoaHoc)
        RETURN nd.id AS user_id, kh.id AS course_id, dg.diem AS rating
    """
    with driver.session() as session:
        result = session.run(query)
        return [{"user_id": r["user_id"], "course_id": r["course_id"], "rating": r["rating"]} for r in result]

def create_train_test_split(ratings):
    """Chia tập train/test theo tỷ lệ 80/20 cho mỗi user"""
    random.seed(RANDOM_SEED)
    user_ratings = defaultdict(list)
    for r in ratings:
        user_ratings[r["user_id"]].append((r["course_id"], r["rating"]))

    train, test = {}, {}
    for uid, items in user_ratings.items():
        if len(items) < MIN_RATINGS:
            continue
            
        random.shuffle(items)
        n_test = max(1, int(len(items) * TEST_RATIO))
        
        test[uid] = {cid for cid, _ in items[:n_test]}
        train[uid] = {cid for cid, _ in items[n_test:]}

    return train, test

def fetch_recommendations_from_neo4j(driver, user_id, train_course_ids, model_type="HYBRID", top_k=10):
    """Lấy gợi ý từ Neo4j bằng Cypher (Đã chặn Data Leakage qua $trainIds)"""
    if model_type == "HYBRID":
        query = """
            MATCH (nd:NguoiDung {id: $userId})-[dg:DANH_GIA]->(kh:KhoaHoc)
            WHERE kh.id IN $trainIds
            WITH kh, (dg.diem / 5.0) AS normalizedRating
            ORDER BY normalizedRating DESC
            LIMIT 5

            MATCH (kh)-[rel:CONTENT_SIMILAR]-(q:KhoaHoc)
            WHERE NOT q.id IN $trainIds

            OPTIONAL MATCH (aiDo:NguoiDung)-[dg_q:DANH_GIA]->(q)
            WITH q, rel.score AS contentScore, normalizedRating, 
                 count(dg_q) AS soLuongDanhGia, coalesce(q.danhGiaTrungBinh, 0.0) AS saoTrungBinh
            WHERE soLuongDanhGia > 0

            WITH q, soLuongDanhGia, saoTrungBinh,
                 (contentScore * 0.4) + (normalizedRating * 0.2) + ((saoTrungBinh / 5.0) * 0.2) + (log10(soLuongDanhGia + 1) * 0.2) AS simScore

            WITH q.id AS courseId, max(simScore) AS finalScore
            ORDER BY finalScore DESC
            LIMIT $topK
            RETURN courseId
        """
    elif model_type == "PURE_CONTENT":
        query = """
            MATCH (nd:NguoiDung {id: $userId})-[dg:DANH_GIA]->(kh:KhoaHoc)
            WHERE kh.id IN $trainIds AND dg.diem >= 3.0
            WITH kh
            MATCH (kh)-[rel:CONTENT_SIMILAR]-(q:KhoaHoc)
            WHERE NOT q.id IN $trainIds
            WITH q.id AS courseId, max(rel.score) AS finalScore
            ORDER BY finalScore DESC
            LIMIT $topK
            RETURN courseId
        """

    with driver.session() as session:
        result = session.run(query, userId=user_id, trainIds=list(train_course_ids), topK=top_k)
        return [r["courseId"] for r in result]

def evaluate(predictions, test_dict, all_course_ids, k_values=TOP_K_VALUES):
    """Tính các metric: NDCG, Precision, Recall, Coverage"""
    course_to_idx = {cid: i for i, cid in enumerate(all_course_ids)}
    n_items = len(all_course_ids)
    results = {}
    all_recommended = set()

    for k in k_values:
        ndcg_list, prec_list, rec_list = [], [], []

        for uid, pred_courses in predictions.items():
            true_courses = test_dict.get(uid, set())
            if not true_courses: continue

            pred_k = pred_courses[:k]
            all_recommended.update(pred_courses)

            hits = len(set(pred_k) & true_courses)
            prec_list.append(hits / k if k > 0 else 0)
            rec_list.append(hits / len(true_courses) if len(true_courses) > 0 else 0)

            pred_idx = [course_to_idx[c] for c in pred_courses if c in course_to_idx]
            true_idx = {course_to_idx[c] for c in true_courses if c in course_to_idx}

            true_rel = np.zeros(n_items)
            for idx in true_idx: true_rel[idx] = 1.0

            pred_sc = np.zeros(n_items)
            for rank, idx in enumerate(pred_idx):
                pred_sc[idx] = len(pred_idx) - rank

            try:
                if len(pred_idx) > 0 and len(true_idx) > 0:
                    score = ndcg_score([true_rel], [pred_sc], k=k)
                    ndcg_list.append(score)
            except Exception: pass

        results[f"ndcg@{k}"] = float(np.mean(ndcg_list)) if ndcg_list else 0.0
        results[f"precision@{k}"] = float(np.mean(prec_list)) if prec_list else 0.0
        results[f"recall@{k}"] = float(np.mean(rec_list)) if rec_list else 0.0

    results["coverage"] = len(all_recommended) / n_items if n_items > 0 else 0.0
    return results

def random_baseline(test_dict, all_course_ids, k=10):
    """Sinh gợi ý ngẫu nhiên để làm baseline so sánh (Giống hàm compare_to_random_baseline của ALS)"""
    random.seed(RANDOM_SEED)
    random_predictions = {}
    
    # Generate random recommendations cho tất cả user được test
    for uid in test_dict.keys():
        # Xáo trộn danh sách course ngẫu nhiên và lấy k khóa học
        random_predictions[uid] = random.sample(all_course_ids, min(k, len(all_course_ids)))
        
    return evaluate(random_predictions, test_dict, all_course_ids, k_values=[5, 10])

def main():
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    
    print("\n1. Đang tải dữ liệu từ Neo4j...")
    ratings = fetch_all_ratings(driver)
    all_course_ids = sorted({r["course_id"] for r in ratings})
    
    print("2. Đang tạo Train/Test Split (80/20)...")
    train_dict, test_dict = create_train_test_split(ratings)
    
    sample_users = random.sample(list(test_dict.keys()), min(1000, len(test_dict)))
    test_sampled = {uid: test_dict[uid] for uid in sample_users} # Tập test của những user được lấy mẫu
    
    predictions_hybrid = {}
    predictions_content = {}
    
    print(f"\n3. Tiến hành tạo gợi ý qua Cypher cho {len(sample_users)} Users...")
    for uid in tqdm(sample_users):
        train_ids = train_dict.get(uid, set())
        if not train_ids: continue
            
        predictions_hybrid[uid] = fetch_recommendations_from_neo4j(
            driver, uid, train_ids, model_type="HYBRID", top_k=10
        )
        
        predictions_content[uid] = fetch_recommendations_from_neo4j(
            driver, uid, train_ids, model_type="PURE_CONTENT", top_k=10
        )

    driver.close()

    print("\n4. ĐÁNH GIÁ CÁC MÔ HÌNH...")
    res_hybrid = evaluate(predictions_hybrid, test_sampled, all_course_ids)
    res_content = evaluate(predictions_content, test_sampled, all_course_ids)

    print("\n5. TÍNH TOÁN ĐƯỜNG CƠ SỞ NGẪU NHIÊN (RANDOM BASELINE)...")
    res_random = random_baseline(test_sampled, all_course_ids, k=10)

    print("\n" + "="*85)
    print(f"{'Metric':<15} | {'Random Baseline':<18} | {'Pure Content':<18} | {'Hybrid (Profile)':<18}")
    print("="*85)
    
    metrics = ["ndcg@5", "ndcg@10", "precision@5", "precision@10", "recall@5", "recall@10", "coverage"]
    for m in metrics:
        val_random = res_random.get(m, 0.0)
        val_content = res_content.get(m, 0.0)
        val_hybrid = res_hybrid.get(m, 0.0)
        print(f"{m:<15} | {val_random:<18.4f} | {val_content:<18.4f} | {val_hybrid:<18.4f}")
    
    print("="*85)
    
    # Tính toán mức độ cải thiện (Improvement) so với Random (dựa trên ndcg@10)
    rand_ndcg10 = res_random.get("ndcg@10", 0.0001) # tránh chia cho 0
    hybrid_ndcg10 = res_hybrid.get("ndcg@10", 0.0)
    improvement = hybrid_ndcg10 / rand_ndcg10 if rand_ndcg10 > 0 else 0
    
    print(f"\n=> Đánh giá: Mô hình Hybrid tốt hơn {improvement:.2f}x lần so với Random Baseline (tính theo NDCG@10)")
    print("=> Hãy lấy bảng này so sánh với kết quả của ALS trong Tutorial 2.")

if __name__ == "__main__":
    main()


1. Đang tải dữ liệu từ Neo4j...
2. Đang tạo Train/Test Split (80/20)...

3. Tiến hành tạo gợi ý qua Cypher cho 1000 Users...


100%|██████████| 1000/1000 [00:08<00:00, 116.44it/s]



4. ĐÁNH GIÁ CÁC MÔ HÌNH...

5. TÍNH TOÁN ĐƯỜNG CƠ SỞ NGẪU NHIÊN (RANDOM BASELINE)...

Metric          | Random Baseline    | Pure Content       | Hybrid (Profile)  
ndcg@5          | 0.0006             | 0.0007             | 0.0058            
ndcg@10         | 0.0018             | 0.0023             | 0.0059            
precision@5     | 0.0006             | 0.0004             | 0.0028            
precision@10    | 0.0009             | 0.0011             | 0.0014            
recall@5        | 0.0008             | 0.0011             | 0.0055            
recall@10       | 0.0039             | 0.0048             | 0.0055            
coverage        | 0.9987             | 0.6141             | 0.5567            

=> Đánh giá: Mô hình Hybrid tốt hơn 3.35x lần so với Random Baseline (tính theo NDCG@10)
=> Hãy lấy bảng này so sánh với kết quả của ALS trong Tutorial 2.


In [2]:
"""
eval_all_models.py
==================
Đánh giá so sánh 4 mô hình gợi ý trên cùng một mốc train/test cố định.

Mô hình:
  1. Random Baseline
  2. ALS            ← dùng als_model.pkl / als_results.pkl
  3. Pure Content   ← Neo4j Cypher
  4. Hybrid         ← Neo4j Cypher

Mốc cố định: train_matrix.npz / test_matrix.npz (không tạo lại split)
"""

import pickle
import warnings
import random

import numpy as np
import scipy.sparse as sp
from neo4j import GraphDatabase
from sklearn.metrics import ndcg_score
from tqdm import tqdm

warnings.filterwarnings("ignore")

# ================================================================
# CẤU HÌNH — chỉnh ở đây nếu cần
# ================================================================
PATHS = {
    "als_model"    : r"D:\KLKS\Recommendation\als_model.pkl",
    "als_results"  : r"D:\KLKS\Recommendation\als_results.pkl",
    "item_mapping" : r"D:\KLKS\processed\item_mapping.pkl",
    "user_mapping" : r"D:\KLKS\processed\user_mapping.pkl",
    "train_matrix" : r"D:\KLKS\processed\train_matrix.npz",
    "test_matrix"  : r"D:\KLKS\processed\test_matrix.npz",
}

NEO4J = {
    "uri"      : "neo4j://127.0.0.1:7687",
    "user"     : "neo4j",
    "password" : "12345678",
}

RANDOM_SEED  = 42
TOP_K_VALUES = [5, 10]
SAMPLE_USERS = 1000   # số user lấy mẫu để đánh giá


# ================================================================
# UTILS
# ================================================================

def load_mapping(path: str):
    """
    Load file pkl chứa mapping. Hỗ trợ cả dict lồng dict (nested) và dict phẳng.
    Trả về: (id_to_idx, idx_to_id)
    """
    with open(path, "rb") as f:
        raw = pickle.load(f)

    # 1. FIX LỖI Ở ĐÂY: Bóc tách trực tiếp nếu là Dict lồng Dict (Dữ liệu của bạn)
    if isinstance(raw, dict):
        if "user_to_idx" in raw and "idx_to_user" in raw:
            return raw["user_to_idx"], raw["idx_to_user"]
        
        if "product_to_idx" in raw and "idx_to_product" in raw:
            return raw["product_to_idx"], raw["idx_to_product"]

    # 2. Logic dự phòng nếu file thực sự là Dict phẳng
    first_key = next(iter(raw))
    if isinstance(first_key, int):
        idx_to_id = raw
        id_to_idx = {v: k for k, v in raw.items()}
    else:
        id_to_idx = raw
        idx_to_id = {v: k for k, v in raw.items()}

    return id_to_idx, idx_to_id


def matrix_to_dict(matrix: sp.spmatrix, idx_to_user: dict, idx_to_item: dict) -> dict:
    """Sparse matrix (CSR) → {user_id: set(item_ids)}"""
    csr    = matrix.tocsr()
    result = {}
    for u_idx in range(csr.shape[0]):
        row = csr.getrow(u_idx)
        if row.nnz == 0:
            continue
        uid = idx_to_user.get(u_idx)
        if uid is None:
            continue
        result[uid] = {idx_to_item[i] for i in row.indices if i in idx_to_item}
    return result


# ================================================================
# EVALUATE
# ================================================================

def evaluate(predictions: dict, test_dict: dict, all_course_ids: list,
             k_values=TOP_K_VALUES) -> dict:
    """
    Tính NDCG@k, Precision@k, Recall@k, Coverage.
    predictions : {user_id: [course_id, ...]}  (sorted by score)
    test_dict   : {user_id: set(course_id)}
    """
    course_to_idx = {cid: i for i, cid in enumerate(all_course_ids)}
    n_items       = len(all_course_ids)
    results       = {}
    all_recommended = set()

    for k in k_values:
        ndcg_list, prec_list, rec_list = [], [], []

        for uid, pred in predictions.items():
            true_set = test_dict.get(uid, set())
            if not true_set:
                continue

            pred_k = pred[:k]
            all_recommended.update(pred)

            hits = len(set(pred_k) & true_set)
            prec_list.append(hits / k)
            rec_list.append(hits / len(true_set))

            # NDCG cần vector đầy đủ
            pred_idx = [course_to_idx[c] for c in pred if c in course_to_idx]
            true_idx = {course_to_idx[c] for c in true_set if c in course_to_idx}

            true_rel = np.zeros(n_items)
            for idx in true_idx:
                true_rel[idx] = 1.0

            pred_sc = np.zeros(n_items)
            for rank, idx in enumerate(pred_idx):
                pred_sc[idx] = len(pred_idx) - rank   # score giảm dần theo rank

            try:
                if pred_idx and true_idx:
                    ndcg_list.append(ndcg_score([true_rel], [pred_sc], k=k))
            except Exception:
                pass

        results[f"ndcg@{k}"]      = float(np.mean(ndcg_list)) if ndcg_list else 0.0
        results[f"precision@{k}"] = float(np.mean(prec_list)) if prec_list else 0.0
        results[f"recall@{k}"]    = float(np.mean(rec_list))  if rec_list  else 0.0

    results["coverage"] = len(all_recommended) / n_items if n_items > 0 else 0.0
    return results


# ================================================================
# MODEL 1 — RANDOM BASELINE
# ================================================================

def get_random_predictions(sample_users, test_dict, all_course_ids, top_k=10):
    random.seed(RANDOM_SEED)
    return {
        uid: random.sample(all_course_ids, min(top_k, len(all_course_ids)))
        for uid in sample_users
        if uid in test_dict
    }


# ================================================================
# MODEL 2 — ALS
# ================================================================

def _try_als_results(idx_to_user, idx_to_item, sample_user_indices, top_k=10):
    """
    Thử dùng als_results.pkl nếu có.
    Hỗ trợ 3 dạng phổ biến:
      A) {user_id_str: [course_id_str, ...]}
      B) {user_idx_int: [(item_idx, score), ...]}
      C) {user_idx_int: [item_idx, ...]}
    Trả về dict predictions hoặc None nếu không dùng được.
    """
    try:
        with open(PATHS["als_results"], "rb") as f:
            res = pickle.load(f)

        if not isinstance(res, dict) or len(res) == 0:
            return None

        first_key = next(iter(res))
        first_val = res[first_key]
        preds     = {}

        # Case A — key là str (user_id), value là list str (course_id)
        if isinstance(first_key, str):
            uid_set = {idx_to_user.get(i) for i in sample_user_indices}
            for uid in uid_set:
                if uid and uid in res:
                    raw = res[uid]
                    preds[uid] = list(raw)[:top_k] if isinstance(raw[0], str) else [
                        idx_to_item[i] for i in raw[:top_k] if i in idx_to_item
                    ]

        # Case B — key là int, value là list of tuple (item_idx, score)
        elif isinstance(first_key, int) and isinstance(first_val, list) and first_val and isinstance(first_val[0], (tuple, list)):
            for u_idx in sample_user_indices:
                uid = idx_to_user.get(u_idx)
                if uid is None or u_idx not in res:
                    continue
                item_ids = [t[0] for t in res[u_idx][:top_k]]
                preds[uid] = [idx_to_item[i] for i in item_ids if i in idx_to_item]

        # Case C — key là int, value là list int (item_idx)
        elif isinstance(first_key, int):
            for u_idx in sample_user_indices:
                uid = idx_to_user.get(u_idx)
                if uid is None or u_idx not in res:
                    continue
                preds[uid] = [idx_to_item[i] for i in res[u_idx][:top_k] if i in idx_to_item]

        if preds:
            print(f"     ✓ Dùng als_results.pkl trực tiếp ({len(preds)} users)")
            return preds

    except Exception as e:
        print(f"     ! als_results.pkl không dùng được ({e}), tính từ model...")

    return None


def get_als_predictions(sample_user_indices, idx_to_user, idx_to_item, train_matrix, top_k=10):
    """
    ALS predictions theo thứ tự ưu tiên:
      1. als_results.pkl  (nếu format phù hợp)
      2. model.recommend() của thư viện implicit
      3. Tính thủ công U @ V.T  (fallback cuối)
    """
    # Bước 1: thử results sẵn
    preds = _try_als_results(idx_to_user, idx_to_item, sample_user_indices, top_k)
    if preds:
        return preds

    # Bước 2 & 3: dùng model
    with open(PATHS["als_model"], "rb") as f:
        model = pickle.load(f)

    train_csr = train_matrix.tocsr()
    preds     = {}

    try:
        # --- implicit recommend() ---
        for u_idx in tqdm(sample_user_indices, desc="     ALS"):
            uid = idx_to_user.get(u_idx)
            if uid is None:
                continue
            user_row = train_csr.getrow(u_idx)
            try:
                # implicit >= 0.6: trả về (array_ids, array_scores)
                item_ids, _ = model.recommend(
                    u_idx, user_row, N=top_k, filter_already_liked_items=True
                )
            except TypeError:
                # implicit < 0.6: trả về [(item_id, score), ...]
                recs     = model.recommend(u_idx, train_matrix, N=top_k)
                item_ids = [r[0] for r in recs]

            preds[uid] = [idx_to_item[i] for i in item_ids if i in idx_to_item]

    except Exception as e:
        print(f"\n     ! implicit.recommend() thất bại: {e}")
        print("     → Fallback: dot product thủ công...")
        preds = {}

        U = np.array(model.user_factors)   # (n_users, factors)
        V = np.array(model.item_factors)   # (n_items, factors)

        for u_idx in tqdm(sample_user_indices, desc="     ALS (manual)"):
            uid = idx_to_user.get(u_idx)
            if uid is None or u_idx >= len(U):
                continue

            scores = U[u_idx] @ V.T                          # shape (n_items,)
            train_items = list(train_csr.getrow(u_idx).indices)
            if train_items:
                scores[train_items] = -np.inf               # loại bỏ đã học

            top_items = np.argsort(scores)[::-1][:top_k]
            preds[uid] = [idx_to_item[i] for i in top_items if i in idx_to_item]

    return preds


# ================================================================
# MODEL 3 & 4 — NEO4J
# ================================================================

CYPHER = {
    "PURE_CONTENT": """
        MATCH (nd:NguoiDung {id: $userId})-[dg:DANH_GIA]->(kh:KhoaHoc)
        WHERE kh.id IN $trainIds AND dg.diem >= 3.0
        WITH kh
        MATCH (kh)-[rel:CONTENT_SIMILAR]-(q:KhoaHoc)
        WHERE NOT q.id IN $trainIds
        WITH q.id AS courseId, max(rel.score) AS finalScore
        ORDER BY finalScore DESC
        LIMIT $topK
        RETURN courseId
    """,
    "HYBRID": """
        MATCH (nd:NguoiDung {id: $userId})-[dg:DANH_GIA]->(kh:KhoaHoc)
        WHERE kh.id IN $trainIds
        WITH kh, (dg.diem / 5.0) AS normalizedRating
        ORDER BY normalizedRating DESC
        LIMIT 5

        MATCH (kh)-[rel:CONTENT_SIMILAR]-(q:KhoaHoc)
        WHERE NOT q.id IN $trainIds

        OPTIONAL MATCH (aiDo:NguoiDung)-[dg_q:DANH_GIA]->(q)
        WITH q, rel.score AS contentScore, normalizedRating,
             count(dg_q)                      AS soLuongDanhGia,
             coalesce(q.danhGiaTrungBinh, 0.0) AS saoTrungBinh
        WHERE soLuongDanhGia > 0

        WITH q, soLuongDanhGia, saoTrungBinh,
             (contentScore      * 0.4)
           + (normalizedRating  * 0.2)
           + ((saoTrungBinh / 5.0) * 0.2)
           + (log10(soLuongDanhGia + 1) * 0.2) AS simScore

        WITH q.id AS courseId, max(simScore) AS finalScore
        ORDER BY finalScore DESC
        LIMIT $topK
        RETURN courseId
    """,
}


def get_neo4j_predictions(sample_users, train_dict, model_type: str, top_k=10):
    driver = GraphDatabase.driver(NEO4J["uri"], auth=(NEO4J["user"], NEO4J["password"]))
    query  = CYPHER[model_type]
    preds  = {}

    with driver.session() as session:
        for uid in tqdm(sample_users, desc=f"     {model_type}"):
            train_ids = list(train_dict.get(uid, set()))
            if not train_ids:
                continue
            result   = session.run(query, userId=uid, trainIds=train_ids, topK=top_k)
            preds[uid] = [r["courseId"] for r in result]

    driver.close()
    return preds


# ================================================================
# MAIN
# ================================================================

def print_table(results: dict, metrics: list):
    """In bảng kết quả đẹp"""
    models = list(results.keys())
    col_w  = 14

    # Header
    header = f"{'Metric':<16}" + "".join(f" | {m:<{col_w}}" for m in models)
    sep    = "=" * len(header)
    print(sep)
    print(header)
    print(sep)

    for metric in metrics:
        row = f"{metric:<16}"
        for m in models:
            row += f" | {results[m].get(metric, 0.0):<{col_w}.4f}"
        print(row)

    print(sep)


def main():
    print("\n" + "=" * 65)
    print("   ĐÁNH GIÁ SO SÁNH 4 MÔ HÌNH GỢI Ý KHÓA HỌC")
    print("   Mốc cố định: train_matrix.npz / test_matrix.npz")
    print("=" * 65)

    # ── 1. Load mappings & matrices ──────────────────────────────
    print("\n[1/5] Load mappings & matrices...")
    user_to_idx, idx_to_user = load_mapping(PATHS["user_mapping"])
    item_to_idx, idx_to_item = load_mapping(PATHS["item_mapping"])

    train_matrix = sp.load_npz(PATHS["train_matrix"])
    test_matrix  = sp.load_npz(PATHS["test_matrix"])

    all_course_ids = sorted(idx_to_item.values())   # sort → thứ tự ổn định

    print(f"  → {len(user_to_idx):,} users | {len(all_course_ids):,} courses")
    print(f"  → Train: {train_matrix.nnz:,} tương tác | Test: {test_matrix.nnz:,} tương tác")

    # ── 2. Matrix → dict ─────────────────────────────────────────
    print("\n[2/5] Chuyển matrix → dict...")
    train_dict = matrix_to_dict(train_matrix, idx_to_user, idx_to_item)
    test_dict  = matrix_to_dict(test_matrix,  idx_to_user, idx_to_item)

    valid_users = sorted(set(train_dict) & set(test_dict))
    print(f"  → {len(valid_users):,} users có cả train lẫn test")

    # ── 3. Sample cố định ────────────────────────────────────────
    random.seed(RANDOM_SEED)
    sample_users        = random.sample(valid_users, min(SAMPLE_USERS, len(valid_users)))
    sample_user_indices = [user_to_idx[uid] for uid in sample_users if uid in user_to_idx]
    test_sampled        = {uid: test_dict[uid] for uid in sample_users}

    print(f"  → Sample {len(sample_users)} users (seed={RANDOM_SEED}) — dùng chung cho tất cả models")

    # ── 4. Predictions ───────────────────────────────────────────
    print("\n[3/5] Sinh predictions từ 4 models...")

    print("  [1] Random Baseline...")
    preds_random = get_random_predictions(sample_users, test_sampled, all_course_ids)

    print("  [2] ALS...")
    preds_als = get_als_predictions(
        sample_user_indices, idx_to_user, idx_to_item, train_matrix
    )

    print("  [3] Pure Content (Neo4j)...")
    preds_content = get_neo4j_predictions(sample_users, train_dict, "PURE_CONTENT")

    print("  [4] Hybrid (Neo4j)...")
    preds_hybrid = get_neo4j_predictions(sample_users, train_dict, "HYBRID")

    # ── 5. Evaluate ──────────────────────────────────────────────
    print("\n[4/5] Tính metrics...")
    all_results = {
        "Random"       : evaluate(preds_random,  test_sampled, all_course_ids),
        "ALS"          : evaluate(preds_als,      test_sampled, all_course_ids),
        "Pure Content" : evaluate(preds_content,  test_sampled, all_course_ids),
        "Hybrid"       : evaluate(preds_hybrid,   test_sampled, all_course_ids),
    }

    # ── 6. In kết quả ────────────────────────────────────────────
    print("\n[5/5] KẾT QUẢ:\n")
    metrics = [
        "ndcg@5", "ndcg@10",
        "precision@5", "precision@10",
        "recall@5", "recall@10",
        "coverage",
    ]
    print_table(all_results, metrics)

    # Improvement so với Random
    base = all_results["Random"].get("ndcg@10", 1e-9)
    print("\n=> Improvement so với Random (NDCG@10):")
    for name in ["ALS", "Pure Content", "Hybrid"]:
        val = all_results[name].get("ndcg@10", 0.0)
        print(f"   {name:<15}: {val / base:.2f}x")

    print()


if __name__ == "__main__":
    main()


   ĐÁNH GIÁ SO SÁNH 4 MÔ HÌNH GỢI Ý KHÓA HỌC
   Mốc cố định: train_matrix.npz / test_matrix.npz

[1/5] Load mappings & matrices...
  → 9,424 users | 1,500 courses
  → Train: 100,829 tương tác | Test: 20,600 tương tác

[2/5] Chuyển matrix → dict...
  → 9,424 users có cả train lẫn test
  → Sample 1000 users (seed=42) — dùng chung cho tất cả models

[3/5] Sinh predictions từ 4 models...
  [1] Random Baseline...
  [2] ALS...


     ALS: 100%|██████████| 1000/1000 [00:00<00:00, 2918.41it/s]


  [3] Pure Content (Neo4j)...


     PURE_CONTENT: 100%|██████████| 1000/1000 [00:06<00:00, 151.98it/s]


  [4] Hybrid (Neo4j)...


     HYBRID: 100%|██████████| 1000/1000 [00:05<00:00, 169.38it/s]



[4/5] Tính metrics...

[5/5] KẾT QUẢ:

Metric           | Random         | ALS            | Pure Content   | Hybrid        
ndcg@5           | 0.0015         | 0.6610         | 0.0009         | 0.0053        
ndcg@10          | 0.0032         | 0.6773         | 0.0018         | 0.0059        
precision@5      | 0.0012         | 0.2494         | 0.0006         | 0.0022        
precision@10     | 0.0019         | 0.1374         | 0.0007         | 0.0013        
recall@5         | 0.0017         | 0.6436         | 0.0017         | 0.0051        
recall@10        | 0.0057         | 0.6941         | 0.0033         | 0.0053        
coverage         | 0.9993         | 0.2400         | 0.6207         | 0.5667        

=> Improvement so với Random (NDCG@10):
   ALS            : 209.79x
   Pure Content   : 0.57x
   Hybrid         : 1.83x

